# Extract All Error Mappings

Loads every result file, classifies errors, and writes structured TSVs to `error_mappings/`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/data/jane/convert/math_gender/conversion_test')
BASE = PROJECT_ROOT / 'full_results'
OUT = PROJECT_ROOT / '3_error_taxonomy/error_mappings'
OUT.mkdir(exist_ok=True)

NON_REASONING = ['gpt-4o', 'qwen-coder', 'llama-4', 'gpt-oss-120b']
REASONING = ['gpt-5.2', 'deepseek-v3.1', 'qwen3-235b-thinking', 'claude-haiku-4-5']
ALL_MODELS = NON_REASONING + REASONING

CONDS = {
    'regular':   'results',
    'no_guide':  'results_no_guide',
    'math_only': 'results_math_only',
}

print(f"Models: {ALL_MODELS}")
print(f"Conditions: {list(CONDS.keys())}")

Models: ['gpt-4o', 'qwen-coder', 'llama-4', 'claude-haiku-4-5', 'gpt-oss-120b', 'gpt-5.2', 'deepseek-v3.1', 'qwen3-235b-thinking']
Conditions: ['regular', 'no_guide', 'math_only']


## 1. Load All Results

In [ ]:
# load in desired file. pick full or only integers
df = pd.read_csv('/data/jane/convert/math_gender/conversion_test/2_analysis/results.csv', sep='\t')

Loaded 3,173,160 rows
Models: ['claude-haiku-4-5', 'deepseek-v3.1', 'gpt-4o', 'gpt-5.2', 'gpt-oss-120b', 'llama-4', 'qwen-coder', 'qwen3-235b-thinking']
Domains: ['bits_bytes', 'clothing_sizes_men_pant_size', 'clothing_sizes_men_shoe_size', 'clothing_sizes_women_bra_size', 'clothing_sizes_women_pant_size', 'clothing_sizes_women_shoe_size', 'cooking', 'currency', 'density', 'energy', 'moles_to_particles', 'speed', 'temperature', 'timezone', 'volume']
Conditions: ['math_only', 'no_guide', 'regular']


In [ ]:
# # rename conditions 
# df['condition'] = df['condition'].replace({'math_only': 'math_only', 'no_guide': 'in_domain_no_guide', 'regular': 'in_domain_with_guide'})

## 2. Classify Errors

In [3]:
def classify_error(row):
    """Classify a wrong answer into an error type.

    Categories:
      correct        – loss == 0
      refusal        – model refused (N/A, cannot, etc.)
      rounding       – loss <= 2 (small precision difference)
      decimal_shift  – answer differs by a power of 10
      magnitude_error– answer is >100x or <0.01x the gold
      wrong_factor   – off by a recognizable integer factor
      sign_error     – correct absolute value, wrong sign
      moderate_error – loss 2..100 not fitting above
      large_error    – loss > 100 not fitting above
    """
    loss = row['loss']
    if pd.isna(loss) or loss == 0:
        return 'correct'

    raw = str(row.get('raw_response', ''))
    if any(kw in raw.lower() for kw in ['n/a', 'cannot', 'i cannot', 'unable to', 'not possible', 'impossible']):
        return 'refusal'

    if loss <= 2:
        return 'rounding'

    try:
        gold = float(row['answer'])
        ma = row.get('model_answer', np.nan)
        model_ans = float(ma) if pd.notna(ma) and str(ma) not in ('', 'nan', 'None') else np.nan
        if pd.isna(model_ans) or gold == 0:
            return 'large_error' if loss > 100 else 'moderate_error'

        ratio = model_ans / gold
        abs_ratio = abs(ratio)

        if abs(gold) > 0 and ((model_ans < 0 and gold > 0) or (model_ans > 0 and gold < 0)):
            if abs(abs(model_ans) - abs(gold)) / abs(gold) < 0.05:
                return 'sign_error'

        for power in range(-6, 7):
            if power == 0:
                continue
            if 0.9 < abs_ratio / (10 ** power) < 1.1:
                return 'decimal_shift'

        if abs_ratio > 100 or abs_ratio < 0.01:
            return 'magnitude_error'

        for factor in [2, 3, 4, 5, 6, 8, 12, 16, 48]:
            if 0.95 < abs_ratio / factor < 1.05 or 0.95 < abs_ratio * factor < 1.05:
                return 'wrong_factor'
    except (ValueError, TypeError, ZeroDivisionError):
        pass

    return 'large_error' if loss > 100 else 'moderate_error'


df['error_type'] = df.apply(classify_error, axis=1)

print("Error type distribution:")
print(df['error_type'].value_counts().to_string())
print(f"\nCorrect: {(df['error_type']=='correct').sum():,} / {len(df):,} "
      f"({(df['error_type']=='correct').mean()*100:.1f}%)")

Error type distribution:
error_type
correct            2575410
rounding            198288
moderate_error      140241
decimal_shift       118081
magnitude_error      57749
wrong_factor         44540
large_error          31352
refusal               6820
sign_error             679

Correct: 2,575,410 / 3,173,160 (81.2%)


## 3. Prepare Shared Filters

In [4]:
# No-distractor subset (used by several extractions)
nodist_mask = (
    df['distractor'].astype(str).isin(['null', 'nan', '', 'None'])
    | df['distractor'].isna()
)
df_nd = df[nodist_mask].copy()

# Condition subsets
mo = df[df['condition'] == 'math_only'].copy()
ng = df[df['condition'] == 'no_guide'].copy()
reg = df[df['condition'] == 'regular'].copy()

# No-distractor + no-clothing for math/no-guide comparison
ng_clean = ng[~ng['domain'].str.contains('clothing')].copy()
for d in [mo, ng_clean]:
    mask = d['distractor'].astype(str).isin(['null', 'nan', '', 'None']) | d['distractor'].isna()
    d.drop(d[~mask].index, inplace=True)

mo['key'] = mo['model'] + '|' + mo['domain'] + '|' + mo['number'] + '|' + mo['answer']
ng_clean['key'] = ng_clean['model'] + '|' + ng_clean['domain'] + '|' + ng_clean['number'] + '|' + ng_clean['answer']

# No-distractor subsets of reg/ng for guide comparison
reg_nd = df_nd[df_nd['condition'] == 'regular'].copy()
ng_nd = df_nd[df_nd['condition'] == 'no_guide'].copy()
reg_nd['key'] = reg_nd['model'] + '|' + reg_nd['domain'] + '|' + reg_nd['number'] + '|' + reg_nd['answer']
ng_nd['key'] = ng_nd['model'] + '|' + ng_nd['domain'] + '|' + ng_nd['number'] + '|' + ng_nd['answer']

print(f"No-distractor rows: {len(df_nd):,}")
print(f"Math-only (no dist): {len(mo):,}")
print(f"No-guide (no dist, no clothing): {len(ng_clean):,}")

No-distractor rows: 770,405
Math-only (no dist): 256,228
No-guide (no dist, no clothing): 256,236


## 4. Extract: Error Rate Summary

One row per model × domain × condition with accuracy, error rate, and loss statistics.

In [5]:
summary = df.groupby(['model', 'domain', 'condition']).agg(
    total=('loss', 'size'),
    n_correct=('loss', lambda x: (x == 0).sum()),
    n_wrong=('loss', lambda x: (x > 0).sum()),
    mean_loss=('loss', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
    median_loss=('loss', lambda x: x[x > 0].median() if (x > 0).any() else 0),
    max_loss=('loss', 'max'),
).reset_index()
summary['accuracy_pct'] = (summary['n_correct'] / summary['total'] * 100).round(2)
summary['error_pct'] = (summary['n_wrong'] / summary['total'] * 100).round(2)
summary.to_csv(OUT / 'error_rate_summary.tsv', sep='\t', index=False)
print(f"error_rate_summary.tsv: {len(summary)} rows")
summary.head()

error_rate_summary.tsv: 320 rows


,model,domain,condition,total,n_correct,n_wrong,mean_loss,median_loss,max_loss,accuracy_pct,error_pct
0,claude-haiku-4-5,bits_bytes,math_only,600,564,36,85.929386,1.114551,900.0,94.00,6.00
1,claude-haiku-4-5,bits_bytes,no_guide,600,356,244,3.633129,2.345804,100.0,59.33,40.67
2,claude-haiku-4-5,bits_bytes,regular,600,506,94,32.406857,2.351471,100.0,84.33,15.67
3,claude-haiku-4-5,clothing_sizes_men_pant_size,no_guide,28,15,13,1.000000,1.000000,1.0,53.57,46.43
4,claude-haiku-4-5,clothing_sizes_men_pant_size,regular,28,28,0,0.000000,0.000000,0.0,100.00,0.00


## 5. Extract: Error Type Breakdown

Count of each error type per model × domain × condition.

In [6]:
etype_counts = (
    df[df['error_type'] != 'correct']
    .groupby(['model', 'domain', 'condition', 'error_type'])
    .size()
    .reset_index(name='count')
)
etype_counts.to_csv(OUT / 'error_type_breakdown.tsv', sep='\t', index=False)
print(f"error_type_breakdown.tsv: {len(etype_counts)} rows")
etype_counts.head()

error_type_breakdown.tsv: 1014 rows


,model,domain,condition,error_type,count
0,claude-haiku-4-5,bits_bytes,math_only,decimal_shift,11
1,claude-haiku-4-5,bits_bytes,math_only,magnitude_error,4
2,claude-haiku-4-5,bits_bytes,math_only,moderate_error,1
3,claude-haiku-4-5,bits_bytes,math_only,rounding,20
4,claude-haiku-4-5,bits_bytes,no_guide,magnitude_error,1


## 6. Extract: Math-Only Correct → No-Guide Wrong

Every case where the model solved the pure arithmetic but failed the domain-phrased version.

In [7]:
mo_correct_keys = set(mo[mo['loss'] == 0]['key'].values)
ng_wrong = ng_clean[ng_clean['loss'] > 0].copy()
ng_wrong['math_was_correct'] = ng_wrong['key'].isin(mo_correct_keys)
disc = ng_wrong[ng_wrong['math_was_correct']].copy()

disc_out = disc[['model', 'domain', 'number', 'answer', 'loss', 'error_type',
                 'prompt', 'raw_response']].copy()
disc_out['raw_response'] = disc_out['raw_response'].str[:500]
disc_out['prompt'] = disc_out['prompt'].str[:300]
disc_out.to_csv(OUT / 'math_correct_noguide_wrong.tsv', sep='\t', index=False)
print(f"math_correct_noguide_wrong.tsv: {len(disc_out):,} rows")

# Summary per model × domain
disc_summary = disc.groupby(['model', 'domain']).agg(
    n_discrepant=('loss', 'size'),
    mean_loss=('loss', 'mean'),
    median_loss=('loss', 'median'),
).reset_index()
ng_totals = ng_clean.groupby(['model', 'domain']).size().reset_index(name='ng_total')
disc_summary = disc_summary.merge(ng_totals, on=['model', 'domain'], how='left')
disc_summary['disc_pct'] = (disc_summary['n_discrepant'] / disc_summary['ng_total'] * 100).round(2)
disc_summary.to_csv(OUT / 'math_vs_noguide_summary.tsv', sep='\t', index=False)
print(f"math_vs_noguide_summary.tsv: {len(disc_summary)} rows")
disc_summary

math_correct_noguide_wrong.tsv: 121,038 rows
math_vs_noguide_summary.tsv: 75 rows


,model,domain,n_discrepant,mean_loss,median_loss,ng_total,disc_pct
0,claude-haiku-4-5,bits_bytes,216,3.858675,2.351471,600,36.00
1,claude-haiku-4-5,cooking,29,32.286313,0.155587,600,4.83
2,claude-haiku-4-5,currency,9717,759587.331076,99.482482,11200,86.76
3,claude-haiku-4-5,density,33,34.309260,1.599427,600,5.50
4,claude-haiku-4-5,energy,24,13.386107,0.137845,1000,2.40
...,...,...,...,...,...,...,...
70,qwen3-235b-thinking,moles_to_particles,54,6.302810,0.367656,400,13.50
71,qwen3-235b-thinking,speed,49,476.442365,97.222220,1200,4.08
72,qwen3-235b-thinking,temperature,7,0.197462,0.181818,1200,0.58
73,qwen3-235b-thinking,timezone,1405,69.288256,30.000000,10800,13.01


## 7. Extract: Cross-Domain Errors

Same input number correct in domain A, wrong in domain B (no-distractor only).

In [8]:
cross_rows = []
for model in df_nd['model'].unique():
    for cond in ['regular', 'no_guide', 'math_only']:
        sub = df_nd[(df_nd['model'] == model) & (df_nd['condition'] == cond)]
        if len(sub) == 0:
            continue
        correct = sub[sub['loss'] == 0]
        wrong = sub[sub['loss'] > 2]

        correct_nums = correct.groupby('number')['domain'].apply(set).to_dict()

        for _, w in wrong.iterrows():
            num = w['number']
            if num in correct_nums:
                c_doms = correct_nums[num] - {w['domain']}
                if c_doms:
                    cross_rows.append({
                        'model': model,
                        'condition': cond,
                        'number': num,
                        'wrong_domain': w['domain'],
                        'wrong_gold': w['answer'],
                        'wrong_loss': w['loss'],
                        'wrong_error_type': w['error_type'],
                        'correct_domains': ','.join(sorted(c_doms)[:5]),
                        'n_correct_domains': len(c_doms),
                        'wrong_response_snippet': str(w['raw_response'])[:200],
                        'wrong_prompt_snippet': str(w['prompt'])[:200],
                    })

cross_df = pd.DataFrame(cross_rows)
cross_df.to_csv(OUT / 'cross_domain_errors.tsv', sep='\t', index=False)
print(f"cross_domain_errors.tsv: {len(cross_df):,} rows")

cross_summary = cross_df.groupby(['model', 'condition', 'wrong_domain']).agg(
    n_cross_errors=('wrong_loss', 'size'),
    mean_loss=('wrong_loss', 'mean'),
).reset_index()
# cross_summary.to_csv(OUT / 'cross_domain_summary.tsv', sep='\t', index=False)
print(f"cross_domain_summary.tsv: {len(cross_summary)} rows")

cross_domain_errors.tsv: 146,564 rows
cross_domain_summary.tsv: 183 rows


## 8. Extract: All Individual Errors

In [9]:
clothing_domains = [d for d in df['domain'].unique() if d.startswith('clothing_sizes_')]
print(f"Merging {len(clothing_domains)} clothing domains into 'clothing_size': {clothing_domains}")
df.loc[df['domain'].isin(clothing_domains), 'domain'] = 'clothing_size'
print(f"Domains after merge: {sorted(df['domain'].unique())}")

Merging 5 clothing domains into 'clothing_size': ['clothing_sizes_men_pant_size', 'clothing_sizes_men_shoe_size', 'clothing_sizes_women_bra_size', 'clothing_sizes_women_pant_size', 'clothing_sizes_women_shoe_size']
Domains after merge: ['bits_bytes', 'clothing_size', 'cooking', 'currency', 'density', 'energy', 'moles_to_particles', 'speed', 'temperature', 'timezone', 'volume']


In [10]:
# err_cols = ['model', 'domain', 'condition', 'number', 'answer', 'loss', 'error_type', 'distractor']
# if 'model_answer' in df.columns:
#     err_cols.append('model_answer')

all_errors = df[df['error_type'] != 'correct'].copy()
all_errors.to_csv(OUT / 'all_errors.tsv', sep='\t', index=False)
print(f"all_errors columns: {all_errors.columns.tolist()}")
print(f"all_errors.tsv: {len(all_errors):,} rows")

all_errors columns: ['domain', 'distractor', 'prompt', 'number', 'answer', 'difficulty', 'raw_response', 'model_answer', 'loss', 'model', 'condition', 'is_reasoning', 'reasoning_tokens', 'call_seconds', 'error_type']
all_errors.tsv: 597,750 rows


In [11]:
# sample 30 per domain and save to examples.csv

all_errors.groupby('domain').sample(30).to_csv(OUT / 'examples_to_analyze.tsv', sep='\t', index=False)

## 9. Extract: Guide vs No-Guide Comparison

- **guide_helps**: correct with guide, wrong without  
- **guide_hurts**: wrong with guide, correct without

In [ ]:
# Guide helps: with-guide correct, no-guide wrong
reg_correct_keys = set(reg_nd[reg_nd['loss'] == 0]['key'].values)
ng_nd_wrong = ng_nd[ng_nd['loss'] > 0].copy()
ng_nd_wrong['guide_was_correct'] = ng_nd_wrong['key'].isin(reg_correct_keys)
guide_helps = ng_nd_wrong[ng_nd_wrong['guide_was_correct']]

guide_summary = guide_helps.groupby(['model', 'domain']).agg(
    n_guide_helps=('loss', 'size'),
    mean_loss=('loss', 'mean'),
).reset_index()
ng_nd_totals = ng_nd.groupby(['model', 'domain']).size().reset_index(name='ng_total')
guide_summary = guide_summary.merge(ng_nd_totals, on=['model', 'domain'], how='left')
guide_summary['guide_helps_pct'] = (guide_summary['n_guide_helps'] / guide_summary['ng_total'] * 100).round(2)
guide_summary.to_csv(OUT / 'guide_vs_noguide_summary.tsv', sep='\t', index=False)
print(f"guide_vs_noguide_summary.tsv: {len(guide_summary)} rows")

# Guide hurts: with-guide wrong, no-guide correct
ng_correct_keys = set(ng_nd[ng_nd['loss'] == 0]['key'].values)
reg_nd_wrong = reg_nd[reg_nd['loss'] > 0].copy()
reg_nd_wrong['noguide_was_correct'] = reg_nd_wrong['key'].isin(ng_correct_keys)
guide_hurts = reg_nd_wrong[reg_nd_wrong['noguide_was_correct']]

guide_hurts_summary = guide_hurts.groupby(['model', 'domain']).size().reset_index(name='n_guide_hurts')
# guide_hurts_summary.to_csv(OUT / 'guide_hurts_cases.tsv', sep='\t', index=False)
# print(f"guide_hurts_cases.tsv: {len(guide_hurts_summary)} rows")

display(guide_summary)
display(guide_hurts_summary)

## 10. Extract: Distractor Impact

In [ ]:
df['has_distractor'] = ~(
    df['distractor'].astype(str).isin(['null', 'nan', '', 'None'])
    | df['distractor'].isna()
)
domains_with_dist = sorted(df[df['has_distractor']]['domain'].unique())
print(f"Domains with distractors: {domains_with_dist}")

dist_rows = []
for dom in domains_with_dist:
    for m in df['model'].unique():
        for cond in ['regular', 'no_guide']:
            for has_d in [True, False]:
                sub = df[(df['domain'] == dom) & (df['model'] == m)
                         & (df['condition'] == cond) & (df['has_distractor'] == has_d)]
                if len(sub) == 0:
                    continue
                dist_rows.append({
                    'domain': dom, 'model': m, 'condition': cond,
                    'has_distractor': has_d,
                    'n': len(sub),
                    'n_correct': int((sub['loss'] == 0).sum()),
                    'n_wrong': int((sub['loss'] > 0).sum()),
                    'error_pct': round((sub['loss'] > 0).mean() * 100, 2),
                    'mean_loss_when_wrong': round(sub[sub['loss'] > 0]['loss'].mean(), 2)
                                           if (sub['loss'] > 0).any() else 0,
                })

dist_df = pd.DataFrame(dist_rows)
# dist_df.to_csv(OUT / 'distractor_impact.tsv', sep='\t', index=False)
# print(f"distractor_impact.tsv: {len(dist_df)} rows")
dist_df.head(10)

## 11. Summary

Print all output files and key numbers.

In [ ]:
print(f"Output directory: {OUT.resolve()}")
print(f"{'File':<40s} {'Rows':>10s}  {'Size':>8s}")
print('─' * 62)
for f in sorted(OUT.glob('*.tsv')):
    nrows = sum(1 for _ in open(f)) - 1
    size_kb = f.stat().st_size / 1024
    print(f"{f.name:<40s} {nrows:>10,d}  {size_kb:>7.0f} KB")

print(f"\n--- Quick stats ---")
print(f"Total rows loaded:       {len(df):>12,d}")
print(f"Total errors:            {(df['error_type']!='correct').sum():>12,d}")
print(f"Math-correct/NG-wrong:   {len(disc_out):>12,d}")
print(f"Cross-domain errors:     {len(cross_df):>12,d}")
print(f"Guide-helps cases:       {guide_summary['n_guide_helps'].sum():>12,.0f}")
print(f"Guide-hurts cases:       {guide_hurts_summary['n_guide_hurts'].sum():>12,d}")